In [0]:
%sql
CREATE CATALOG IF NOT EXISTS workingspace
MANAGED LOCATION 'abfss://workingspace@practicestorage863.dfs.core.windows.net'
COMMENT "This Catalog is used for the revison of Delta Lake"

In [0]:
%sql
USE CATALOG workingspace;

CREATE SCHEMA IF NOT EXISTS bronze
COMMENT "This schema is used for the bronze layer"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workingspace.bronze.orders
(
    order_id     BIGINT,
    customer_id  BIGINT,
    order_status STRING,
    order_amount DECIMAL(10,2),
    country      STRING,
    order_date   DATE,
    create_at    TIMESTAMP_NTZ

)
USING DELTA
PARTITIONED BY (country)
COMMENT 'This table is used for the bronze layer in workingspace'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
);


In [0]:
%sql
INSERT INTO bronze.orders VALUES
(1, 101, 'PLACED',    250.00, 'IN', '2026-07-20', current_timestamp()),
(2, 102, 'SHIPPED',   540.50, 'US', '2026-07-21', current_timestamp()),
(3, 103, 'PLACED',    120.00, 'IN', '2026-07-22', current_timestamp()),
(4, 104, 'CANCELLED', 300.00, 'UK', '2026-07-22', current_timestamp());

In [0]:
%sql
DESC history workingspace.bronze.orders

In [0]:
%sql
SHOW TBLPROPERTIES workingspace.bronze.orders

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workingspace.bronze.orders_test (
    order_id      BIGINT,
    customer_id   BIGINT,
    order_status  STRING,
    order_amount  DECIMAL(10,2),
    country       STRING,
    order_date    DATE,
    created_at    TIMESTAMP
)
USING DELTA
COMMENT 'This is the delta used to see the DEFAULT TBLPROPERTIES';

In [0]:
%sql
DESC HISTORY workingspace.bronze.orders_test;

In [0]:
%sql
SHOW TBLPROPERTIES workingspace.bronze.orders_test;

In [0]:
%sql
INSERT OVERWRITE workingspace.bronze.orders
PARTITION (country = 'IN')
(order_id, customer_id, order_status, order_amount, order_date, create_at)
VALUES
(1, 101, 'DELIVERED', 250.00, '2026-07-20', current_timestamp()),
(3, 103, 'DELIVERED', 120.00, '2026-07-22', current_timestamp());

In [0]:
%sql
DESCRIBE history workingspace.bronze.orders;

In [0]:
%sql
select * from workingspace.bronze.orders;

In [0]:
%sql

INSERT OVERWRITE workingspace.bronze.orders_test
select * from workingspace.bronze.orders;

In [0]:
%sql
select * from workingspace.bronze.orders_test;
    
-- DESCRIBE HISTORY workingspace.bronze.orders_test;
    
-- SHOW TBLPROPERTIES workingspace.bronze.orders_test;
    

In [0]:
%sql
DESC extended workingspace.bronze.orders;

In [0]:
%sql
DESC HISTORY workingspace.bronze.orders;

In [0]:
%sql
SELECT * FROM workingspace.bronze.orders VERSION AS OF 2 where country = "IN";

In [0]:
%sql
create table IF NOT EXISTS workingspace.bronze.orders_dev
SHALLOW CLONE workingspace.bronze.orders;

In [0]:
%sql
CREATE table IF NOT EXISTS workingspace.bronze.orders_dev_deep
DEEP CLONE workingspace.bronze.orders
LOCATION 'abfss://workingspace@practicestorage863.dfs.core.windows.net/bronze/'

In [0]:
%sql

desc extended workingspace.bronze.orders_dev_deep;


In [0]:
%sql

DESC EXTENDED workingspace.bronze.orders_dev;